# Multi-corpus Llama-3.1 judge (batched Transformers)

Judges ~7k texts with the same rubric as Shrishti **`step3f_llm_judge.py`**, writes **`step3h`**-style `clean_dataset/` CSVs.

| Corpus | Cap | HF / Drive source |
|--------|-----|-------------------|
| CNN/Daily Mail | 2000 | `cnn_dailymail` 3.0.0 train |
| XSum | 2000 | `xsum` train (`document`) |
| CoQA | 2000 | `stanfordnlp/coqa` train stories |
| WeeBit | 500 | Drive `data_train.csv` (Kaggle) |
| CommonLit | 500 | Drive `commonlit_train.csv` or HF |

**Uses batched `transformers` generate** (reliable on Colab; no vLLM).

## Shrishti references
- Judge: `logistic-regression/llm_as_a_judge/llm_as_a_judge/scripts/step3f_llm_judge.py`
- Clean CSVs: `step3h_build_clean_dataset.py`
- Template: `trail/CoQA_CNN_Llama_Judge_Colab.ipynb`

## Run order
1. Config → Install → HF login → Drive  
2. Prepare splits (~10 min)  
3. Judge batched (~55 min on A100 with `JUDGE_BATCH_SIZE=8`)  
4. Build clean_dataset  

Outputs: `DRIVE_ROOT/{splits,llm_judge,clean_dataset,manifest.json}`

In [ ]:
# ── Config ─────────────────────────────────────────────────────────────────
MAX_CNN_ARTICLES = 2000
MAX_XSUM_DOCS = 2000
MAX_COQA_STORIES = 2000
MAX_WEEBIT_ROWS = 500
MAX_COMMONLIT_ROWS = 500
MIN_CHARS = 80
SEED = 42

JUDGE_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
MAX_MODEL_LEN = 2048
MAX_NEW_TOKENS = 8
JUDGE_BATCH_SIZE = 8  # lower to 4 if OOM on T4/L4

WORK_ROOT = "/content/trail_multi_judge_work"
SPLITS_DIR = f"{WORK_ROOT}/splits"
JUDGE_DIR = f"{WORK_ROOT}/llm_judge"
CLEAN_DIR = f"{WORK_ROOT}/clean_dataset"
HF_CACHE_DIR = f"{WORK_ROOT}/hf_cache"
LOGS_DIR = f"{WORK_ROOT}/logs"

DRIVE_ROOT = "/content/drive/MyDrive/BeyondFK/trail/judge_multi_corpus"

# WeeBit raw Kaggle CSV (sample 500 from train)
WEE_BIT_TRAIN_CSV = "/content/drive/MyDrive/BeyondFK/clean_dataset/data_train.csv"

# CommonLit: upload competition train.csv or set HF path
COMMONLIT_CSV = "/content/drive/MyDrive/BeyondFK/trail/commonlit/train.csv"
USE_HF_COMMONLIT = False  # if True, tries HF when CSV missing

SKIP_CORPORA = []  # e.g. ["weebit", "commonlit"] to skip missing files

CORPORA = [
    {"split_name": "cnn_dailymail_2k", "source_dataset": "cnn_dailymail", "subject": "news", "clean_filename": "cnn_dailymail_2k.csv"},
    {"split_name": "xsum_2k", "source_dataset": "xsum", "subject": "news", "clean_filename": "xsum_2k.csv"},
    {"split_name": "coqa_2k", "source_dataset": "coqa", "subject": "coqa", "clean_filename": "coqa_2k.csv"},
    {"split_name": "weebit_500", "source_dataset": "weebit", "subject": "reading", "clean_filename": "weebit_500.csv"},
    {"split_name": "commonlit_500", "source_dataset": "commonlit", "subject": "reading", "clean_filename": "commonlit_500.csv"},
]

In [ ]:
!pip install -q "datasets>=2.18" pandas huggingface_hub codecarbon
!pip install -q "transformers>=4.44" accelerate bitsandbytes

import os
for d in (WORK_ROOT, SPLITS_DIR, JUDGE_DIR, CLEAN_DIR, HF_CACHE_DIR, LOGS_DIR):
    os.makedirs(d, exist_ok=True)
os.environ["HF_HOME"] = HF_CACHE_DIR
print("Install OK")

In [ ]:
from huggingface_hub import login
login()
print("HF login OK")

In [ ]:
from google.colab import drive
import os

drive.mount("/content/drive")
for sub in ("splits", "llm_judge", "clean_dataset", "logs"):
    os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)
print("Drive:", DRIVE_ROOT)

In [ ]:
# ── Rubric + helpers (step3f / CoQA_CNN notebook) ───────────────────────────
import hashlib
import json
import re
import shutil
import time

import pandas as pd
from datasets import load_dataset

RUBRIC = (
    "Classify the following text by its target reader's US education level.\n"
    "Choose exactly one of:\n"
    "- elementary  (US grades 1-5, simple vocabulary, short sentences)\n"
    "- middle      (US grades 6-8)\n"
    "- high        (US grades 9-12, advanced vocabulary, complex ideas)\n\n"
    "Text:\n{text}\n\n"
    "Reply with one word only: elementary, middle, or high."
)

CORE_COLS = [
    "split", "orig_split", "orig_idx", "source_dataset", "subject", "raw_label",
    "full_text", "education_level_original", "education_level_judge", "judge_raw_response",
]

TEXT_CANDIDATES = ["full_text", "text", "passage", "article", "content", "story", "document", "excerpt"]
LABEL_CANDIDATES = ["label", "y", "target", "readability", "level", "class", "labels", "grade"]

WEE_BIT_INT_MAP = {0: "elementary", 1: "elementary", 2: "middle", 3: "middle", 4: "high"}
WEE_BIT_STR_MAP = {
    "0": "elementary", "1": "elementary", "2": "middle", "3": "middle", "4": "high",
    "wrlevel2": "elementary", "wrlevel3": "elementary", "wrlevel4": "middle",
    "bitks3": "middle", "bitks": "middle", "bitgcse": "high", "gcse": "high",
    "elementary": "elementary", "middle": "middle", "high": "high",
}


def parse_label(raw: str) -> str:
    s = str(raw).strip().lower()
    s = re.sub(r"^[^a-z]+", "", s)
    if s.startswith("elem"):
        return "elementary"
    if s.startswith("mid"):
        return "middle"
    if s.startswith("high"):
        return "high"
    return "elementary"


def _base_row(full_text, split_name, source_dataset, subject, orig_idx, license_id, education_level, raw_label):
    return {
        "full_text": full_text,
        "education_level": education_level,
        "source_dataset": source_dataset,
        "subject": subject,
        "raw_label": str(raw_label),
        "split": split_name,
        "orig_split": split_name,
        "orig_idx": orig_idx,
        "source_license": license_id,
    }


def _pick_column(df, candidates, kind="column"):
    cols = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in cols:
            return cols[cand.lower()]
    raise ValueError(f"No {kind} in {list(df.columns)}; tried {candidates}")


def weebit_label_to_level(raw) -> str:
    if pd.isna(raw):
        return "middle"
    if isinstance(raw, (int, float)) and not isinstance(raw, bool):
        return WEE_BIT_INT_MAP.get(int(raw), "middle")
    s = str(raw).strip().lower().replace(" ", "").replace("-", "")
    if s in WEE_BIT_STR_MAP:
        return WEE_BIT_STR_MAP[s]
    if s.isdigit():
        return WEE_BIT_INT_MAP.get(int(s), "middle")
    for k, v in WEE_BIT_STR_MAP.items():
        if k in s:
            return v
    return "middle"


def commonlit_target_to_level(target: float, q33: float, q67: float) -> str:
    if target <= q33:
        return "elementary"
    if target <= q67:
        return "middle"
    return "high"

In [ ]:
# ── Step 1: Prepare splits ─────────────────────────────────────────────────

def _write_split(rows, path):
    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise RuntimeError(f"no rows for {path}")
    df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)
    df["orig_idx"] = range(len(df))
    df.to_csv(path, index=False)
    print(f"wrote {len(df)} -> {path}")
    return path


def prepare_cnn():
    sn, path = "cnn_dailymail_2k", f"{SPLITS_DIR}/cnn_dailymail_2k.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"[cnn] reuse {len(pd.read_csv(path))} rows")
        return path
    ds = load_dataset("cnn_dailymail", "3.0.0", split="train")
    rows, seen = [], set()
    for ex in ds:
        t = str(ex.get("article") or "").strip()
        if len(t) < MIN_CHARS:
            continue
        k = hashlib.sha256(t.encode()).hexdigest()
        if k in seen:
            continue
        seen.add(k)
        rows.append(_base_row(t, sn, "cnn_dailymail", "news", len(rows), "cnn_dailymail/3.0.0", "middle", ""))
        if len(rows) >= MAX_CNN_ARTICLES:
            break
    return _write_split(rows, path)


def prepare_xsum():
    sn, path = "xsum_2k", f"{SPLITS_DIR}/xsum_2k.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"[xsum] reuse {len(pd.read_csv(path))} rows")
        return path
    ds = load_dataset("xsum", split="train")
    rows, seen = [], set()
    for ex in ds:
        t = str(ex.get("document") or "").strip()
        if len(t) < MIN_CHARS:
            continue
        k = hashlib.sha256(t.encode()).hexdigest()
        if k in seen:
            continue
        seen.add(k)
        rows.append(_base_row(t, sn, "xsum", "news", len(rows), "xsum", "middle", ""))
        if len(rows) >= MAX_XSUM_DOCS:
            break
    return _write_split(rows, path)


def prepare_coqa():
    sn, path = "coqa_2k", f"{SPLITS_DIR}/coqa_2k.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"[coqa] reuse {len(pd.read_csv(path))} rows")
        return path
    ds = load_dataset("stanfordnlp/coqa", split="train")
    rows, seen = [], set()
    for ex in ds:
        t = str(ex.get("story") or "").strip()
        if len(t) < MIN_CHARS:
            continue
        k = hashlib.sha256(t.encode()).hexdigest()
        if k in seen:
            continue
        seen.add(k)
        rows.append(_base_row(t, sn, "coqa", "coqa", len(rows), "stanfordnlp/coqa", "middle", ""))
        if len(rows) >= MAX_COQA_STORIES:
            break
    return _write_split(rows, path)


def prepare_weebit():
    sn, path = "weebit_500", f"{SPLITS_DIR}/weebit_500.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"[weebit] reuse {len(pd.read_csv(path))} rows")
        return path
    if not os.path.exists(WEE_BIT_TRAIN_CSV):
        raise FileNotFoundError(f"Upload WeeBit train CSV to {WEE_BIT_TRAIN_CSV}")
    df_in = pd.read_csv(WEE_BIT_TRAIN_CSV)
    text_col = _pick_column(df_in, TEXT_CANDIDATES, "text")
    label_col = _pick_column(df_in, LABEL_CANDIDATES, "label")
    rows = []
    for _, r in df_in.iterrows():
        t = str(r[text_col]).strip()
        if len(t) < MIN_CHARS:
            continue
        raw = r[label_col]
        lvl = weebit_label_to_level(raw)
        rows.append(_base_row(t, sn, "weebit", "reading", len(rows), "kaggle/weebit", lvl, raw))
        if len(rows) >= MAX_WEEBIT_ROWS:
            break
    return _write_split(rows, path)


def prepare_commonlit():
    sn, path = "commonlit_500", f"{SPLITS_DIR}/commonlit_500.csv"
    if os.path.exists(path) and os.path.getsize(path) > 0:
        print(f"[commonlit] reuse {len(pd.read_csv(path))} rows")
        return path
    if os.path.exists(COMMONLIT_CSV):
        df_in = pd.read_csv(COMMONLIT_CSV)
    elif USE_HF_COMMONLIT:
        try:
            df_in = load_dataset("commonlit/commonlit", split="train").to_pandas()
        except Exception:
            df_in = load_dataset("muennighoff/commonlit-readability", split="train").to_pandas()
    else:
        raise FileNotFoundError(f"Set COMMONLIT_CSV or USE_HF_COMMONLIT=True")
    text_col = _pick_column(df_in, ["excerpt", "text", "full_text"], "text")
    target_col = _pick_column(df_in, ["target", "readability", "label"], "target")
    targets = pd.to_numeric(df_in[target_col], errors="coerce")
    q33, q67 = targets.quantile(0.33), targets.quantile(0.67)
    rows = []
    for i, r in df_in.iterrows():
        t = str(r[text_col]).strip()
        if len(t) < MIN_CHARS or pd.isna(targets.iloc[i]):
            continue
        tgt = float(targets.iloc[i])
        lvl = commonlit_target_to_level(tgt, q33, q67)
        rows.append(_base_row(t, sn, "commonlit", "reading", len(rows), "commonlit", lvl, tgt))
        if len(rows) >= MAX_COMMONLIT_ROWS:
            break
    return _write_split(rows, path)


split_paths = {}
builders = {
    "cnn_dailymail": prepare_cnn,
    "xsum": prepare_xsum,
    "coqa": prepare_coqa,
    "weebit": prepare_weebit,
    "commonlit": prepare_commonlit,
}
name_map = {
    "cnn_dailymail": "cnn_dailymail_2k",
    "xsum": "xsum_2k",
    "coqa": "coqa_2k",
    "weebit": "weebit_500",
    "commonlit": "commonlit_500",
}
for key, fn in builders.items():
    if key in SKIP_CORPORA:
        print(f"[skip] {key}")
        continue
    try:
        p = fn()
        split_paths[name_map[key]] = p
        shutil.copy2(p, f"{DRIVE_ROOT}/splits/{name_map[key]}.csv")
    except Exception as e:
        print(f"[FAIL] {key}: {e}")

print(f"Prepared {len(split_paths)} splits")

In [ ]:
# ── Step 2: Batched Llama judge (Transformers) ─────────────────────────────
import torch
from codecarbon import EmissionsTracker
from transformers import AutoModelForCausalLM, AutoTokenizer
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(JUDGE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"  # better for batched generation on decoder-only LMs

print(f"Loading {JUDGE_MODEL} ...")
model = AutoModelForCausalLM.from_pretrained(
    JUDGE_MODEL,
    torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    device_map="auto" if torch.cuda.is_available() else None,
)
if not torch.cuda.is_available():
    model = model.to(device)
model.eval()

# token budget for text inside rubric (step3f)
empty_msgs = [{"role": "user", "content": RUBRIC.format(text="")}]
empty_prompt = tokenizer.apply_chat_template(empty_msgs, tokenize=False, add_generation_prompt=True)
overhead = len(tokenizer(empty_prompt, add_special_tokens=False)["input_ids"])
TEXT_TOKEN_BUDGET = MAX_MODEL_LEN - overhead - MAX_NEW_TOKENS - 8
print(f"chat overhead={overhead} -> max_text_tokens={TEXT_TOKEN_BUDGET}")


def build_prompt(text: str) -> str:
    ids = tokenizer(str(text), add_special_tokens=False)["input_ids"]
    if len(ids) > TEXT_TOKEN_BUDGET:
        text = tokenizer.decode(ids[:TEXT_TOKEN_BUDGET], skip_special_tokens=True)
    msgs = [{"role": "user", "content": RUBRIC.format(text=text)}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)


@torch.inference_mode()
def generate_batch(prompts: list[str]) -> list[str]:
    enc = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=MAX_MODEL_LEN,
    )
    enc = {k: v.to(model.device) for k, v in enc.items()}
    out = model.generate(
        **enc,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    input_lens = enc["attention_mask"].sum(dim=1)
    raws = []
    for j in range(out.shape[0]):
        start = int(input_lens[j].item())
        raws.append(tokenizer.decode(out[j, start:], skip_special_tokens=True))
    return raws


def judge_split_batched(split_name: str) -> str:
    split_csv = f"{SPLITS_DIR}/{split_name}.csv"
    judge_csv = f"{JUDGE_DIR}/{split_name}_judge.csv"
    df = pd.read_csv(split_csv)
    n = len(df)
    if os.path.exists(judge_csv):
        done = pd.read_csv(judge_csv)
        if len(done) == n:
            print(f"[judge] {split_name}: already complete ({n})")
            shutil.copy2(judge_csv, f"{DRIVE_ROOT}/llm_judge/{split_name}_judge.csv")
            return judge_csv

    texts = df["full_text"].astype(str).tolist()
    print(f"[judge] {split_name}: building prompts for {n} texts ...")
    prompts = [build_prompt(t) for t in tqdm(texts, desc="prompts")]

    raws = []
    t0 = time.time()
    for i in tqdm(range(0, len(prompts), JUDGE_BATCH_SIZE), desc=f"judge {split_name}"):
        raws.extend(generate_batch(prompts[i : i + JUDGE_BATCH_SIZE]))
    labels = [parse_label(r) for r in raws]

    rec = pd.DataFrame({
        "orig_split": df["orig_split"] if "orig_split" in df.columns else split_name,
        "orig_idx": df["orig_idx"] if "orig_idx" in df.columns else df.index,
        "source_dataset": df["source_dataset"].astype(str),
        "education_level": df["education_level"].astype(str),
        "llm_judge_label": labels,
        "judge_raw_response": raws,
    })
    rec.to_csv(judge_csv, index=False)
    agree = (rec["education_level"] == rec["llm_judge_label"]).mean()
    print(
        f"[judge] {split_name}: {n} rows in {(time.time()-t0)/60:.1f} min | "
        f"agree_orig={agree:.3f} | {dict(rec['llm_judge_label'].value_counts())}"
    )
    shutil.copy2(judge_csv, f"{DRIVE_ROOT}/llm_judge/{split_name}_judge.csv")
    return judge_csv


tracker = EmissionsTracker(project_name="trail_multi_corpus_judge", output_dir=LOGS_DIR, log_level="warning")
tracker.start()
try:
    for cfg in CORPORA:
        sn = cfg["split_name"]
        if cfg["source_dataset"] in SKIP_CORPORA:
            continue
        if sn not in split_paths and not os.path.exists(f"{SPLITS_DIR}/{sn}.csv"):
            print(f"[judge] skip {sn} (split missing)")
            continue
        judge_split_batched(sn)
finally:
    tracker.stop()
print("Judge complete.")

In [ ]:
# ── Step 3: Build clean_dataset (step3h format) ─────────────────────────────
manifest = {"judge_model": JUDGE_MODEL, "corpora": []}

for cfg in CORPORA:
    if cfg["source_dataset"] in SKIP_CORPORA:
        continue
    split_name = cfg["split_name"]
    split_csv = f"{SPLITS_DIR}/{split_name}.csv"
    judge_csv = f"{JUDGE_DIR}/{split_name}_judge.csv"
    if not os.path.exists(split_csv) or not os.path.exists(judge_csv):
        print(f"[clean] skip {split_name} (missing split or judge)")
        continue
    sdf = pd.read_csv(split_csv)
    jdf = pd.read_csv(judge_csv)
    if len(sdf) != len(jdf):
        raise RuntimeError(f"{split_name}: row mismatch {len(sdf)} vs {len(jdf)}")
    out = pd.DataFrame()
    out["split"] = [split_name] * len(sdf)
    out["orig_split"] = sdf.get("orig_split", split_name)
    out["orig_idx"] = sdf.get("orig_idx", range(len(sdf)))
    out["source_dataset"] = cfg["source_dataset"]
    out["subject"] = cfg["subject"]
    out["raw_label"] = sdf.get("raw_label", "")
    out["full_text"] = sdf["full_text"].astype(str)
    out["education_level_original"] = sdf["education_level"].astype(str)
    out["education_level_judge"] = jdf["llm_judge_label"].astype(str)
    out["judge_raw_response"] = jdf["judge_raw_response"].astype(str)
    out = out[CORE_COLS]
    clean_path = f"{CLEAN_DIR}/{cfg['clean_filename']}"
    out.to_csv(clean_path, index=False)
    drive_clean = f"{DRIVE_ROOT}/clean_dataset/{cfg['clean_filename']}"
    shutil.copy2(clean_path, drive_clean)
    print(f"[{split_name}] n={len(out)} -> {drive_clean}")
    manifest["corpora"].append({
        "split_name": split_name,
        "n_rows": int(len(out)),
        "original_distribution": {
            str(k): int(v) for k, v in out["education_level_original"].value_counts().items()
        },
        "judge_distribution": {
            str(k): int(v) for k, v in out["education_level_judge"].value_counts().items()
        },
        "clean_file": cfg["clean_filename"],
    })

for mp in (f"{DRIVE_ROOT}/manifest.json", f"{WORK_ROOT}/manifest.json"):
    with open(mp, "w", encoding="utf-8") as f:
        json.dump(manifest, f, indent=2)

print("\nDONE. clean_dataset on Drive:")
for cfg in CORPORA:
  if cfg["source_dataset"] not in SKIP_CORPORA:
    print(f"  {DRIVE_ROOT}/clean_dataset/{cfg['clean_filename']}")